# AI Model Validation: From Mathematics to Trust
## Episode 11 — Distribution Shift and Model Drift: When Yesterday's Evidence Stops Being Valid

**Author: Vishwanath Akuthota**  
**Founder of DrPinnacle, OpenVals and The Foundry's**

Episode 10 asked whether a model survives controlled stress. Episode 11 asks a harder deployment question:

> **What happens when the world itself changes after validation?**

A model is validated using evidence collected under particular populations, distributions, time periods, measurement systems and operating conditions. Those conditions are not guaranteed to remain stationary.

Central principle:

> **Validation evidence has a context. When the context changes, the evidence must be re-examined.**

This episode develops covariate shift, label shift, concept drift, temporal drift, performance drift, Population Stability Index, KL divergence, Jensen–Shannon divergence, Wasserstein distance, hypothesis testing, multivariate drift, windowing, delayed labels, drift versus performance degradation, false alarms, retraining triggers and continuous validation.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import platform, sys

rng=np.random.default_rng(1111)

print("Python :",sys.version.split()[0])
print("System :",platform.system(),platform.release())
print("Machine:",platform.machine())
print("NumPy  :",np.__version__)


## 1. Validation Evidence Is Distribution-Dependent

Let development data come from:

### (X,Y) ~ P₀(X,Y)

At deployment time t:

### (X,Y) ~ Pₜ(X,Y)

If:

### Pₜ(X,Y) ≠ P₀(X,Y)

then some assumptions behind the original evaluation have changed.

This does **not** automatically mean the model has failed.

It means the original evidence may no longer describe the current operating environment with the same strength.


## 2. The Joint Distribution Factorization

The joint distribution can be factorized as:

### P(X,Y) = P(Y|X)P(X)

or:

### P(X,Y) = P(X|Y)P(Y)

These factorizations help distinguish different forms of distribution change.

The type of shift matters because different shifts require different diagnostics and responses.


## 3. Covariate Shift

Covariate shift is commonly expressed as:

### Pₜ(X) ≠ P₀(X)

while approximately:

### Pₜ(Y|X) = P₀(Y|X)

The population of inputs changes, while the conditional mapping from X to Y remains approximately stable.

Example: a model begins receiving customers from a different demographic or geographic mixture while the underlying outcome mechanism remains similar.


## 4. Label Shift

Label shift is commonly described as:

### Pₜ(Y) ≠ P₀(Y)

while approximately:

### Pₜ(X|Y) = P₀(X|Y)

The prevalence of outcomes changes.

Examples can include fraud prevalence, disease prevalence, default rates, defect rates or incident frequency changing over time.


## 5. Concept Drift

Concept drift changes the conditional relationship:

### Pₜ(Y|X) ≠ P₀(Y|X)

This is particularly important because the relationship learned by the model itself may no longer represent reality.

### Same Input ≠ Same Outcome Relationship

Concept drift can invalidate a model even if the marginal input distribution appears relatively stable.


## 6. Drift Taxonomy Is Not Perfectly Universal

Terms such as **data drift**, **concept drift**, **prior probability shift**, **virtual drift** and **dataset shift** are used somewhat differently across literature and industry.

Scientific validation should therefore record the mathematical condition being tested rather than relying only on a label.

### Define the Distributional Change, Not Just Its Name


## 7. Controlled Temporal Experiment

We construct a simple binary system.

At baseline:

### X ~ N(0,1)

and:

### P(Y=1|X=x) = sigmoid(1.5x − 0.4)

We then create different deployment changes.


In [ ]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

n=6000
x0=rng.normal(0,1,n)
p0=sigmoid(1.5*x0-.4)
y0=rng.binomial(1,p0)

print("Baseline input mean:",x0.mean())
print("Baseline prevalence:",y0.mean())


## 8. Covariate Shift Experiment

Change the distribution of X while preserving the conditional probability mechanism.


In [ ]:
x_cov=rng.normal(1.0,1.2,n)
p_cov=sigmoid(1.5*x_cov-.4)
y_cov=rng.binomial(1,p_cov)

print("Baseline mean X :",x0.mean())
print("Shifted mean X  :",x_cov.mean())
print("Baseline Y rate :",y0.mean())
print("Shifted Y rate  :",y_cov.mean())

plt.figure(figsize=(8,5))
plt.hist(x0,bins=50,density=True,alpha=.5,label="Baseline")
plt.hist(x_cov,bins=50,density=True,alpha=.5,label="Deployment")
plt.xlabel("X")
plt.ylabel("Density")
plt.title("Covariate Shift")
plt.legend()
plt.show()


## 9. Concept Drift Experiment

Keep the input distribution approximately stable but change the outcome relationship:

Baseline:

### P₀(Y=1|X=x) = sigmoid(1.5x − 0.4)

Deployment:

### Pₜ(Y=1|X=x) = sigmoid(0.55x + 0.2)


In [ ]:
x_concept=rng.normal(0,1,n)
p_concept=sigmoid(.55*x_concept+.2)
y_concept=rng.binomial(1,p_concept)

grid=np.linspace(-3,3,300)

plt.figure(figsize=(8,5))
plt.plot(grid,sigmoid(1.5*grid-.4),label="Baseline P(Y=1|X)")
plt.plot(grid,sigmoid(.55*grid+.2),label="Drifted P(Y=1|X)")
plt.xlabel("X")
plt.ylabel("Probability of Y=1")
plt.title("Concept Drift")
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 10. Data Drift Does Not Equal Model Failure

Suppose:

### Pₜ(X) ≠ P₀(X)

This tells us the input population changed.

It does **not** logically imply:

### Performanceₜ < Performance₀

A model can experience measurable data drift without material performance degradation.

Conversely, concept drift may degrade performance even when simple input-distribution monitoring misses the problem.

### Drift ≠ Failure
### No Detected Drift ≠ Guaranteed Safety


## 11. Population Stability Index — PSI

For bins i with baseline proportion pᵢ and current proportion qᵢ:

### PSI = Σᵢ (qᵢ − pᵢ) ln(qᵢ / pᵢ)

PSI is widely used as an operational distribution-stability statistic.

It depends on binning and requires care when proportions approach zero.

There is no scientifically universal PSI threshold that is correct for every model and application.


In [ ]:
def psi(reference,current,bins=10):
    edges=np.quantile(reference,np.linspace(0,1,bins+1))
    edges[0],edges[-1]=-np.inf,np.inf
    p=np.histogram(reference,bins=edges)[0]/len(reference)
    q=np.histogram(current,bins=edges)[0]/len(current)
    eps=1e-8
    p=np.clip(p,eps,None)
    q=np.clip(q,eps,None)
    return np.sum((q-p)*np.log(q/p))

for shift in [0,.2,.5,1.0,1.5]:
    current=rng.normal(shift,1,n)
    print(f"Mean shift={shift:>3}: PSI={psi(x0,current):.4f}")


## 12. PSI Is a Signal, Not a Verdict

PSI depends on:

- reference population
- bin definitions
- sample size
- variable distribution
- time window

Therefore:

### PSI Value ≠ Universal Business Meaning

A threshold should be validated against model sensitivity, operational risk and historical behaviour rather than copied mechanically.


## 13. Kullback–Leibler Divergence

For discrete distributions P and Q:

### D_KL(P || Q) = Σᵢ P(i) ln[P(i)/Q(i)]

For continuous densities:

### D_KL(P || Q) = ∫ p(x) ln[p(x)/q(x)] dx

Properties:

### D_KL(P||Q) ≥ 0

but generally:

### D_KL(P||Q) ≠ D_KL(Q||P)

KL divergence is therefore not a metric distance.


## 14. KL Divergence and Support

If Q assigns zero probability where P assigns positive probability, KL divergence can become infinite.

Empirical estimation therefore requires careful density estimation, smoothing or discretization.

A numerical KL value is inseparable from the estimation procedure used to obtain it.


In [ ]:
def hist_probs(a,bins):
    h,_=np.histogram(a,bins=bins)
    p=h.astype(float)+1e-10
    return p/p.sum()

edges=np.linspace(-5,6,80)
P=hist_probs(x0,edges)
Q=hist_probs(x_cov,edges)

kl_pq=np.sum(P*np.log(P/Q))
kl_qp=np.sum(Q*np.log(Q/P))

print("KL(P||Q):",kl_pq)
print("KL(Q||P):",kl_qp)


## 15. Jensen–Shannon Divergence

Define mixture:

### M = (P + Q)/2

Then:

### JS(P,Q) = 1/2 D_KL(P||M) + 1/2 D_KL(Q||M)

Unlike KL divergence, Jensen–Shannon divergence is symmetric:

### JS(P,Q) = JS(Q,P)

and remains finite under common discrete formulations.

Its square root is a metric under standard conditions.


In [ ]:
M=.5*(P+Q)
js=.5*np.sum(P*np.log(P/M))+.5*np.sum(Q*np.log(Q/M))
print("Jensen-Shannon divergence:",js)
print("sqrt(JS):",np.sqrt(js))


## 16. Wasserstein Distance

In one dimension, the first Wasserstein distance can be written using quantile functions:

### W₁(P,Q) = ∫₀¹ |F_P⁻¹(u) − F_Q⁻¹(u)| du

It has an intuitive interpretation as the amount of probability mass movement required to transform one distribution into another.

Unlike KL-based measures, it incorporates the geometry of the sample space.


In [ ]:
def wasserstein_1d(a,b,points=5000):
    u=np.linspace(.0001,.9999,points)
    qa=np.quantile(a,u)
    qb=np.quantile(b,u)
    return np.mean(np.abs(qa-qb))

for shift in [0,.25,.5,1,1.5]:
    cur=rng.normal(shift,1,n)
    print(f"Mean shift={shift:>4}: W1≈{wasserstein_1d(x0,cur):.4f}")


## 17. Different Drift Metrics Answer Different Questions

PSI measures binned population redistribution.

KL divergence measures relative information divergence.

Jensen–Shannon provides a symmetric divergence.

Wasserstein distance incorporates movement through the feature space.

### No Single Drift Metric Is Universally Best

Metric choice should match the data type, dimensionality, operational meaning and failure mechanism.


## 18. Statistical Hypothesis Testing for Drift

A two-sample test can formalize:

### H₀: P₀ = Pₜ

against:

### H₁: P₀ ≠ Pₜ

But a small p-value is not the same as operationally important drift.

With very large samples, tiny distribution differences can become statistically detectable.

### Statistical Significance ≠ Operational Significance


## 19. Kolmogorov–Smirnov Distance

For empirical cumulative distribution functions Fₙ and Gₘ:

### D_n,m = sup_x |Fₙ(x) − Gₘ(x)|

The statistic captures the maximum vertical separation between the two empirical CDFs.

Here we compute the distance directly without requiring an external statistics package.


In [ ]:
def ks_distance(a,b):
    values=np.sort(np.concatenate([a,b]))
    aa=np.sort(a); bb=np.sort(b)
    Fa=np.searchsorted(aa,values,side="right")/len(aa)
    Fb=np.searchsorted(bb,values,side="right")/len(bb)
    return np.max(np.abs(Fa-Fb))

for shift in [0,.1,.25,.5,1]:
    cur=rng.normal(shift,1,n)
    print(f"shift={shift:<4} KS distance={ks_distance(x0,cur):.4f}")


## 20. Sample Size Changes Detectability

A fixed distributional difference becomes easier to detect as sample size grows.

This is useful scientifically, but it means drift monitoring cannot rely only on significance testing.

Evidence should combine:

### Statistical Signal + Effect Size + Model Sensitivity + Operational Consequence


## 21. Univariate Drift ≠ Multivariate Drift

Monitoring each feature independently can miss changes in relationships between features.

It is possible for:

### P₀(X_j) ≈ Pₜ(X_j)

for every individual feature j,

while:

### P₀(X₁,...,X_d) ≠ Pₜ(X₁,...,X_d)

because dependencies or correlations changed.


In [ ]:
n2=5000
base=rng.multivariate_normal([0,0],[[1,.1],[.1,1]],n2)
drift=rng.multivariate_normal([0,0],[[1,.9],[.9,1]],n2)

print("Feature 1 mean:",base[:,0].mean(),drift[:,0].mean())
print("Feature 2 mean:",base[:,1].mean(),drift[:,1].mean())
print("Baseline correlation:",np.corrcoef(base.T)[0,1])
print("Drifted correlation :",np.corrcoef(drift.T)[0,1])

plt.figure(figsize=(7,6))
plt.scatter(base[:600,0],base[:600,1],s=10,alpha=.35,label="Baseline")
plt.scatter(drift[:600,0],drift[:600,1],s=10,alpha=.35,label="Deployment")
plt.xlabel("X1"); plt.ylabel("X2")
plt.title("Multivariate Drift Hidden by Similar Marginals")
plt.legend()
plt.grid(alpha=.2)
plt.show()


## 22. Representation Drift

For high-dimensional data, raw-feature monitoring may be insufficient.

Let a representation function h map input x into latent representation z:

### Z = h(X)

We may monitor:

### P₀(Z) vs Pₜ(Z)

This can reveal changes in semantic or learned feature space.

However, representation-based drift depends on the stability and meaning of h itself.


## 23. Prediction Drift

When labels are delayed or unavailable, monitor model outputs:

### P₀(Ŷ) vs Pₜ(Ŷ)

or probabilistic scores:

### P₀(p̂) vs Pₜ(p̂)

Prediction drift can be an early warning signal.

But:

### Prediction Drift ≠ Proven Performance Drift

The output distribution may change because the real population legitimately changed.


## 24. Performance Drift

Once labels become available:

### Mₜ = Metric(Yₜ, Ŷₜ)

Performance drift can be expressed:

### ΔMₜ = Mₜ − M₀

This directly addresses model behaviour, but labels may arrive slowly, selectively or with measurement error.


In [ ]:
def accuracy(y,p,threshold=.5):
    return np.mean((p>=threshold)==y)

# Baseline model remains fixed.
pred0=sigmoid(1.5*x0-.4)
pred_concept=sigmoid(1.5*x_concept-.4)

print("Baseline accuracy:",accuracy(y0,pred0))
print("After concept drift:",accuracy(y_concept,pred_concept))


## 25. Calibration Drift

Episode 09 established:

### P(Y=1 | p̂=p) ≈ p

Calibration drift means this relationship changes with time:

### Calibration_t₀(p) ≠ Calibration_t₁(p)

A model can retain useful ranking while its probability interpretation deteriorates.

Therefore performance monitoring should not be reduced to accuracy alone.


## 26. Temporal Windows

Monitoring compares data across time windows.

Reference window:

### W_ref = [t₀, t₁]

Current window:

### W_cur = [t₂, t₃]

Window size creates a trade-off:

### Small Window → Faster Detection + Higher Variance

### Large Window → Greater Stability + Slower Detection

There is no universally correct window length.


## 27. Sliding and Tumbling Windows

**Tumbling windows** are non-overlapping periods.

**Sliding windows** move continuously and often overlap.

Sliding windows can detect changes more responsively, but successive estimates are correlated because they reuse observations.

The monitoring design must therefore be treated as part of the statistical procedure.


## 28. Gradual, Sudden, Recurring and Incremental Drift

Drift can occur in different temporal patterns:

### Sudden Drift — rapid transition
### Gradual Drift — old and new concepts coexist temporarily
### Incremental Drift — progressive movement
### Recurring Drift — previously observed conditions return

The temporal pattern influences detection and response strategy.


In [ ]:
T=120
time=np.arange(T)

sudden=np.where(time<60,0,1.0)
gradual=np.clip((time-40)/40,0,1)
recurring=.5+.5*np.sin(time/12)

plt.figure(figsize=(9,5))
plt.plot(time,sudden,label="Sudden")
plt.plot(time,gradual,label="Gradual")
plt.plot(time,recurring,label="Recurring")
plt.xlabel("Time")
plt.ylabel("Illustrative concept parameter")
plt.title("Different Temporal Drift Patterns")
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 29. Drift Detection as Sequential Evidence

At each time t, monitoring produces evidence Eₜ.

A naive rule might trigger whenever:

### Dₜ > τ

where Dₜ is a drift statistic and τ is a threshold.

But repeated testing over time increases opportunities for false alarms.

Monitoring therefore requires alert design, persistence rules, multiple-testing awareness or sequential methods where appropriate.


## 30. Persistence Rules

Instead of alerting on one threshold crossing, a system might require:

### Dₜ > τ for k consecutive windows

or combine severity with persistence.

This reduces sensitivity to transient noise but delays detection.

### Faster Detection ↔ False Alarm Control


## 31. Drift Thresholds Are Decision Thresholds

A drift threshold τ is not a law of nature.

It encodes a trade-off between:

- missed drift
- false alerts
- monitoring cost
- retraining cost
- business risk
- safety consequence

Therefore:

### Drift Threshold = Statistical Evidence + Operational Decision


## 32. Delayed Labels

Many production systems receive Y long after prediction.

At time t:

### Xₜ and Ŷₜ are available now

but:

### Yₜ becomes available at t + Δ

This creates two monitoring layers:

**Early signals:** input drift, representation drift, prediction drift.

**Confirmed signals:** performance drift, calibration drift, failure rate.

The distinction should remain explicit.


## 33. Data Quality Can Mimic Drift

A changed distribution may result from:

- sensor failure
- schema changes
- unit conversion
- missing values
- pipeline bugs
- duplicate records
- upstream product changes

Before concluding that the world changed, validate that the measurement system did not change incorrectly.

### Observed Drift = Possible Reality Change + Possible Data-System Change


## 34. Seasonality Is Not Necessarily Harmful Drift

Some populations naturally vary by:

- hour
- weekday
- month
- season
- campaign
- geography
- economic cycle

A monitoring system that ignores known seasonality can repeatedly raise false alarms.

Reference distributions should reflect the actual temporal structure of deployment.


## 35. Drift Can Be Benign

Suppose customer mix changes, but model performance remains stable or improves.

Then drift is real, but model degradation is not.

This is why a mature validation system separates:

### Distribution Change
### Performance Change
### Risk Change

These are related but not identical.


## 36. Model Sensitivity to Drift

Not every feature drift matters equally.

Suppose feature X_j changes strongly but the model barely uses it.

Another feature may drift only slightly but sit near a sensitive decision boundary.

Therefore drift severity should be interpreted together with model sensitivity.

### Large Data Drift ≠ Large Model Impact


## 37. Drift-to-Performance Relationship

Let Dₜ denote drift magnitude and ΔMₜ performance change.

A useful empirical question is:

### How strongly does Dₜ predict ΔMₜ?

Historical monitoring data can help identify which drift measures and features actually anticipate model degradation.

This is stronger than assigning arbitrary thresholds without evidence.


## 38. Retraining Is Not the Automatic Answer

When drift is detected, possible responses include:

- investigate data quality
- collect labels
- recalibrate probabilities
- adjust thresholds
- restrict operating domain
- increase human review
- retrain
- replace the model
- temporarily disable automation

### Drift Detected ≠ Retrain Immediately

Response should match the diagnosed failure mechanism.


## 39. Continuous Validation Trigger

A continuous validation system can combine evidence:

### Trigger = f(Data Drift, Performance Drift, Calibration Drift, Failure Rate, Risk)

A trigger should lead first to a defined validation or operational action, not blindly to model retraining.


## 40. Evidence Expiration

Suppose validation was performed at t₀.

It does not logically follow that:

### Validated(t₀) ⇒ Validated(t₁)

for arbitrary future t₁.

A more defensible interpretation is:

### Validation Evidence = Evidence under stated conditions and time context

This does not mean evidence expires on a universal fixed date. It means continued applicability depends on whether relevant assumptions remain sufficiently stable.


## 41. OpenVals Perspective

A continuous validation platform should preserve:

- reference dataset and time period
- current monitoring window
- feature distributions
- prediction distributions
- outcome prevalence
- PSI
- KL / JS divergence where appropriate
- Wasserstein distance where appropriate
- statistical test results
- effect sizes
- multivariate drift indicators
- model performance over time
- calibration over time
- subgroup drift
- data-quality incidents
- alert threshold and persistence logic
- label delay
- validation trigger
- investigation outcome
- retraining/recalibration decision
- model version
- post-change evidence

The purpose is not to produce more alerts.

It is to determine when yesterday's evidence no longer deserves today's confidence.


## 42. Scientific Drift Chain

### Reference Evidence → Current Population → Distribution Comparison → Model Impact → Uncertainty → Risk Assessment → Validation Trigger → Action → New Evidence

This is the bridge from one-time model evaluation to continuous AI assurance.


## 43. Mathematical Summary

### Pₜ(X,Y) ≠ P₀(X,Y)

### P(X,Y) = P(Y|X)P(X)

### Covariate Shift: Pₜ(X) ≠ P₀(X)

### Label Shift: Pₜ(Y) ≠ P₀(Y)

### Concept Drift: Pₜ(Y|X) ≠ P₀(Y|X)

### PSI = Σᵢ(qᵢ−pᵢ)ln(qᵢ/pᵢ)

### D_KL(P||Q) = ΣᵢP(i)ln[P(i)/Q(i)]

### JS(P,Q) = ½D_KL(P||M) + ½D_KL(Q||M)

### W₁(P,Q) = ∫₀¹|F_P⁻¹(u)−F_Q⁻¹(u)|du

### D_KS = sup_x|Fₙ(x)−Gₘ(x)|

### ΔMₜ = Mₜ − M₀

### Calibrationₜ(p) = Pₜ(Y=1|p̂=p)

### Trigger = f(Drift, Performance, Calibration, Failure, Risk)


## 44. Central Conclusion

### Yesterday's Evidence ≠ Today's Guarantee

> **A model does not become untrustworthy merely because the data changed. But when the conditions supporting its evidence change, trust must be re-measured.**

Drift detection is not the final objective.

The scientific objective is to determine:

### What changed?
### How much did it change?
### Does the model care?
### Did performance or calibration change?
### Is the change operationally important?
### What action is justified by the evidence?

This leads to a larger OpenVals principle:

> **Validation should not be treated as an event. For deployed AI, validation becomes a continuous evidence process.**

## Next Episode

**Episode 12 — Reliability and Failure Analysis: How Often, Where, and Why Does an AI System Fail?**

Episode 12 can develop reliability mathematically through failure probability, conditional failure rates, survival-style reasoning, MTBF/MTTF concepts where applicable, confidence intervals, failure taxonomy, fault trees, error clustering, correlated failures, rare-event evidence, reliability curves and continuous assurance.
